In [ ]:
# import SparkSession
from pyspark.sql import SparkSession
# create SparkSession Object
spark = SparkSession.builder.master("local[*]").appName("readCsvFiles").getOrCreate()

In [24]:
# skipRows Option does not work in my spark version

# df = spark.read.option("skipRows", 1).csv("/Users/shishir/Pyspark/SampleData", header=True)
# df.show()
# df.printSchema()
# df.columns
#   .option("skipRows", 1) 

In [25]:
raw_df = spark.read.csv("/Users/shishir/Pyspark/SampleData")

# Get second row as header
header = raw_df.take(2)[1]

# Remove first two rows
data = raw_df.rdd.zipWithIndex().filter(lambda x: x[1] >= 2).map(lambda x: x[0])
df = spark.createDataFrame(data, schema=header)

df.show()

+----------+----------+----------+---------------------+---------------------------+------------+------------------+------------------------+
| startDate|   endDate|      asin|orderedRevenue_amount|orderedRevenue_currencyCode|orderedUnits|shippedCogs_amount|shippedCogs_currencyCode|
+----------+----------+----------+---------------------+---------------------------+------------+------------------+------------------------+
|2024-11-04|2024-11-04|B0CGY6J3X7|                  0.0|                        USD|           0|                 0|                     USD|
|2024-11-05|2024-11-05|B0CGY6J3X7|                  0.0|                        USD|           0|                 0|                     USD|
|2024-11-06|2024-11-06|B0CGY6J3X7|                  0.0|                        USD|           0|                 0|                     USD|
|2024-11-07|2024-11-07|B0CGY6J3X7|                  0.0|                        USD|           0|                 0|                     USD|
|2024-

In [32]:
# import everything from types module
from pyspark.sql.types import * 

# defined a schema which is of StructType
schema = StructType([StructField("name",StringType()), StructField("gender",StringType()), StructField("age",IntegerType()), StructField("Salary",IntegerType())]) 

# read csv file into spark df with enforced schema. 
df = spark.read.csv("/Users/shishir/Pyspark/SampleData/Employee1.csv", schema = schema, header=True)
df.show()
df.printSchema()

+------+------+---+------+
|  name|gender|age|Salary|
+------+------+---+------+
|  Ajay|  Male| 52|100000|
|Rasmhi|Female| 46|200000|
+------+------+---+------+

root
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Salary: integer (nullable = true)



26/01/12 11:00:28 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: name , gender, age, salary
 Schema: name, gender, age, Salary
Expected: name but found: name 
CSV file: file:///Users/shishir/Pyspark/SampleData/Employee1.csv


In [35]:
# If you want to read multiple files, you can pass list of path strings: files that you want to load
# Ofcourse, all the files should follow the same structure
# reading 2 csv files into spark df. 
df = spark.read.csv(["/Users/shishir/Pyspark/SampleData/Employee1.csv", "/Users/shishir/Pyspark/SampleData/Employee2.csv"], header=True)
df.show()
df.printSchema() # since schema was not explicilty defined, spark automatically inferred the schema and bcoz it is reading from csv, spark treats everything as string

+-------+------+---+------+
|  name |gender|age|salary|
+-------+------+---+------+
|Shishir|  Male| 52|500000|
|Anshika|Female| 46| 20000|
|   Ajay|  Male| 52|100000|
| Rasmhi|Female| 46|200000|
+-------+------+---+------+

root
 |-- name : string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: string (nullable = true)



In [38]:
# If you want to read multiple files, you can pass list of path strings: files that you want to load
# Ofcourse, all the files should follow the same structure
# you can also read all the csv files within a directory.
df = spark.read.csv("/Users/shishir/Pyspark/SampleData")
df.show()
df.printSchema() # since schema was not explicilty defined, spark automatically inferred the schema and bcoz it is reading from csv, spark treats everything as string

+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                 _c0|             _c1|                 _c2|                 _c3|                 _c4|                 _c5|                 _c6|                 _c7|
+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|reportType=GET_VE...|reportPeriod=DAY|sellingProgram=BU...|distributorView=M...|lastUpdatedDate=2...|dataStartTime=202...|dataEndTime=2024-...|marketplaceIds=AT...|
|           startDate|         endDate|                asin|orderedRevenue_am...|orderedRevenue_cu...|        orderedUnits|  shippedCogs_amount|shippedCogs_curre...|
|          2024-11-04|      2024-11-04|          B0CGY6J3X7|                 0.0|                 USD|                   0|                   0|                 USD|
|   

In [39]:
# There are 2 ways to read/load csv files or other types of files like (json, parquet etc) into spark df
# spark.read.csv("filepath")
# or 
# spark.read.format("csv").load("filepath")
# spark.read returns a DataFrameReader object. This object exposes several methods allowing you to load data from files
# Anyway, you can use help() to refer to the doc anytime 
help(spark.read)

Help on DataFrameReader in module pyspark.sql.readwriter object:

class DataFrameReader(OptionUtils)
 |  DataFrameReader(spark: 'SparkSession')
 |  
 |  Interface used to load a :class:`DataFrame` from external storage systems
 |  (e.g. file systems, key-value stores, etc). Use :attr:`SparkSession.read`
 |  to access this.
 |  
 |  .. versionadded:: 1.4.0
 |  
 |  .. versionchanged:: 3.4.0
 |      Supports Spark Connect.
 |  
 |  Method resolution order:
 |      DataFrameReader
 |      OptionUtils
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, spark: 'SparkSession')
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  csv(self, path: Union[str, List[str]], schema: Union[pyspark.sql.types.StructType, str, NoneType] = None, sep: Optional[str] = None, encoding: Optional[str] = None, quote: Optional[str] = None, escape: Optional[str] = None, comment: Optional[str] = None, header: Union[bool, str, NoneType] = None, inferSchema: Unio

In [ ]:
help(spark.read)

In [9]:

df = spark.createDataFrame([{"name":"Shishir", "age":23}])
help(df.write)

Help on DataFrameWriter in module pyspark.sql.readwriter object:

class DataFrameWriter(OptionUtils)
 |  DataFrameWriter(df: 'DataFrame')
 |  
 |  Interface used to write a :class:`DataFrame` to external storage systems
 |  (e.g. file systems, key-value stores, etc). Use :attr:`DataFrame.write`
 |  to access this.
 |  
 |  .. versionadded:: 1.4.0
 |  
 |  .. versionchanged:: 3.4.0
 |      Supports Spark Connect.
 |  
 |  Method resolution order:
 |      DataFrameWriter
 |      OptionUtils
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, df: 'DataFrame')
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  bucketBy(self, numBuckets: int, col: Union[str, List[str], Tuple[str, ...]], *cols: Optional[str]) -> 'DataFrameWriter'
 |      Buckets the output by the given columns. If specified,
 |      the output is laid out on the file system similar to Hive's bucketing scheme,
 |      but with a different bucket hash function and is not 

In [16]:
spark.createDataFrame([
(100, "Hyukjin Kwon"), (120, "Hyukjin Kwon"), (140, "Haejoon Lee")],
schema=["age", "name"]
).write.bucketBy(1, "name").sortBy("age").mode(
"overwrite").saveAsTable("sorted_bucketed_table")

In [17]:
spark.sql("Select * from sorted_bucketed_table").show()

+---+------------+
|age|        name|
+---+------------+
|100|Hyukjin Kwon|
|120|Hyukjin Kwon|
|140| Haejoon Lee|
+---+------------+



In [ ]:
help(spark.read)